# Research paper exploration

Querying everything `save_papers` and `research_brief` have written to `company_context_documents` (`source = 'semantic_scholar'`). Useful for confirming a Slack academic-research turn actually persisted, and for exploring what the agent's `semantic_scholar.search` indexed lane sees. For Slack channel data, see [`slack.ipynb`](slack.ipynb).

Connection plumbing lives in the `centaur_db` package — `db.connect()` starts a kubectl port-forward to the in-cluster Postgres if one isn't already up, and tears it down on kernel shutdown.

Use `db.query(conn, sql, params)` for SELECTs (returns a `pandas.DataFrame`) and `db.execute(conn, sql, params)` for INSERT/UPDATE/DELETE (returns rowcount).

In [ ]:
import centaur_db as db

conn = db.connect()
q = lambda sql, params=None: db.query(conn, sql, params)
q("SELECT current_database() AS db, version() AS pg_version").iloc[0].to_dict()

## Semantic Scholar — saved papers and briefs

Queries `company_context_documents` for everything the `save_papers` and `research_brief` workflows have written (`source = 'semantic_scholar'`). Useful for confirming that a Slack academic-research turn actually persisted, and for exploring what the agent's BM25 retriever sees in the **indexed lane** of `semantic_scholar.search`.

The shape lines up with how the workflows write rows:

- `source_type = 'paper'` — one row per persisted Semantic Scholar paper. `metadata` carries the full S2 record (year, citation count, authors, externalIds) plus the traceability query.
- `source_type = 'research_brief'` — one row per generated brief (markdown in `body`). Child papers link via `parent_document_id` → brief `document_id`; use the join in the section below to list a brief's papers.

In [ ]:
q(
    "SELECT "
    "  source_type, "
    "  left(title, 80) AS title, "
    "  author_name, "
    "  (metadata->>'year')::int AS year, "
    "  (metadata->>'citationCount')::int AS citations, "
    "  length(body) AS body_len, "
    "  parent_document_id IS NOT NULL AS has_parent, "
    "  source_updated_at "
    "FROM company_context_documents "
    "WHERE source = 'semantic_scholar' "
    "ORDER BY source_updated_at DESC "
    "LIMIT 50"
)

### Briefs and their child papers

Each `research_brief` row groups N child paper rows via `parent_document_id`. This view shows every brief on file, the query that triggered it, and how many papers it bundled — a quick way to see what the agent has been researching.

In [ ]:
q(
    "SELECT "
    "  b.metadata->>'query' AS query, "
    "  left(b.title, 60) AS brief_title, "
    "  count(p.document_id) AS paper_count, "
    "  b.source_updated_at "
    "FROM company_context_documents b "
    "LEFT JOIN company_context_documents p "
    "  ON p.parent_document_id = b.document_id "
    "  AND p.source = 'semantic_scholar' "
    "  AND p.source_type = 'paper' "
    "WHERE b.source = 'semantic_scholar' "
    "  AND b.source_type = 'research_brief' "
    "GROUP BY b.document_id, b.title, b.metadata, b.source_updated_at "
    "ORDER BY b.source_updated_at DESC"
)

### Drill into one brief's papers

Child papers link to their brief via `parent_document_id` (authoritative). Both brief and papers also store the original query in `metadata->>'query'` for filtering. Set `BRIEF_QUERY` to a substring of the query you care about, then run the join below.

In [ ]:
BRIEF_QUERY = "active inference"

q(
    """
    SELECT
      p.title,
      p.author_name,
      (p.metadata->>'year')::int AS year,
      (p.metadata->>'citationCount')::int AS citations,
      p.metadata->>'paperId' AS paper_id,
      p.parent_document_id IS NOT NULL AS has_parent,
      p.source_updated_at
    FROM company_context_documents b
    JOIN company_context_documents p
      ON p.parent_document_id = b.document_id
    WHERE b.source = 'semantic_scholar'
      AND b.source_type = 'research_brief'
      AND b.metadata->>'query' ILIKE %s
      AND p.source = 'semantic_scholar'
      AND p.source_type = 'paper'
    ORDER BY (p.metadata->>'citationCount')::int DESC NULLS LAST
    """,
    (f"%{BRIEF_QUERY}%",),
)

Same papers filtered by `metadata->>'query'` on the paper rows alone. Useful when comparing against orphan papers saved before brief linking (`has_parent = false`).

In [ ]:
q(
    """
    SELECT
      title,
      author_name,
      (metadata->>'year')::int AS year,
      (metadata->>'citationCount')::int AS citations,
      parent_document_id IS NOT NULL AS has_parent,
      source_updated_at
    FROM company_context_documents
    WHERE source = 'semantic_scholar'
      AND source_type = 'paper'
      AND metadata->>'query' ILIKE %s
    ORDER BY source_updated_at DESC
    """,
    (f"%{BRIEF_QUERY}%",),
)

### Brief markdown body

The rendered lit-review lives in the brief row's `body` column (same markdown the agent can post to Slack).

In [ ]:
brief = q(
    """
    SELECT title, body, metadata->>'query' AS query, source_updated_at
    FROM company_context_documents
    WHERE source = 'semantic_scholar'
      AND source_type = 'research_brief'
      AND metadata->>'query' ILIKE %s
    ORDER BY source_updated_at DESC
    LIMIT 1
    """,
    (f"%{BRIEF_QUERY}%",),
)
brief.iloc[0]["body"] if len(brief) else "No brief matched BRIEF_QUERY"

### BM25 over saved Semantic Scholar rows

This is exactly what the agent's `semantic_scholar.search` indexed lane runs under the hood — same `body @@@` operator, same `paradedb.score` ranking. Run it locally to see what BM25 hits a query produces against the cache, before the agent ever calls the live S2 API.

In [ ]:
q(
    "SELECT "
    "  source_type, "
    "  left(title, 60) AS title, "
    "  paradedb.score(document_id) AS score, "
    "  (metadata->>'year')::int AS year, "
    "  left(body, 240) AS preview "
    "FROM company_context_documents "
    "WHERE source = 'semantic_scholar' "
    "  AND body @@@ %s "
    "ORDER BY score DESC LIMIT 10",
    ("attention mechanism transformer",),
)

## Archived full-text PDFs

The `archive_papers` workflow (and `research_brief` with `archive=true`) writes **three rows** per paper, joined by Semantic Scholar `paperId`:

1. **`company_context_documents` `source_type='paper'`** — abstract + S2 metadata. Document_id: `semantic_scholar:paper:{paperId}`. Written by every persistence path (`save_papers` / `research_brief` / `archive_papers`).
2. **`company_context_documents` `source_type='paper_fulltext'`** — parsed Markdown body, **capped at 1 MiB UTF-8** for the BM25 index. `parent_document_id` → the `paper` row's document_id. Written only when archiving.
3. **`paper_archives`** — overlay-owned. Raw `pdf_bytes` (BYTEA) + uncapped `parsed_text` + provenance (`source_url`, `mime_type`, `size_bytes`, `pdf_sha256`, `parser_used`, `truncated`). PK on `paper_id`. **Source of truth** — lets us re-render the cc_docs body without re-fetching from the publisher.

The cc_docs `paper_fulltext.metadata->>'pdfSha256'` joins back to `paper_archives.pdf_sha256` for byte-level integrity checks.

### Recently archived papers

One row per archived PDF. `size_mb` is the on-disk bytes; `body_kb` is the parsed Markdown that got indexed for BM25 (capped at 1 MiB).

In [ ]:
q(
    """
    SELECT
      a.paper_id,
      left(coalesce(p.title, '(no cc_docs paper row)'), 70) AS title,
      a.parser_used,
      round(a.size_bytes / 1024.0 / 1024.0, 2) AS size_mb,
      round(length(a.parsed_text) / 1024.0, 1) AS parsed_text_kb,
      a.truncated,
      left(a.pdf_sha256, 12) AS sha256_prefix,
      a.archived_at
    FROM paper_archives a
    LEFT JOIN company_context_documents p
      ON p.document_id = 'semantic_scholar:paper:' || a.paper_id
    ORDER BY a.archived_at DESC
    LIMIT 50
    """
)

### Sanity check: do all three rows exist for each archive?

The workflow upserts `paper`, `paper_fulltext`, and `paper_archives` in sequence. If any of them is missing for a given `paper_id`, something went sideways (transient DB error mid-pipeline, manual delete, etc.). This check shows `has_paper` / `has_fulltext` / `has_archive` for every `paper_id` that appears in *any* of the three; rows where any column is `False` are the ones worth investigating.

In [ ]:
q(
    """
    WITH paper_ids AS (
      SELECT split_part(document_id, ':', 3) AS paper_id
        FROM company_context_documents
       WHERE source = 'semantic_scholar'
         AND source_type IN ('paper', 'paper_fulltext')
      UNION
      SELECT paper_id FROM paper_archives
    ),
    joined AS (
      SELECT
        ids.paper_id,
        (p.document_id IS NOT NULL) AS has_paper,
        (f.document_id IS NOT NULL) AS has_fulltext,
        (a.paper_id    IS NOT NULL) AS has_archive,
        left(coalesce(p.title, f.title, ''), 60) AS title,
        coalesce(a.archived_at, p.source_updated_at) AS last_touched
      FROM paper_ids ids
      LEFT JOIN company_context_documents p
        ON p.document_id = 'semantic_scholar:paper:' || ids.paper_id
      LEFT JOIN company_context_documents f
        ON f.document_id = 'semantic_scholar:paper_fulltext:' || ids.paper_id
      LEFT JOIN paper_archives a
        ON a.paper_id = ids.paper_id
    )
    SELECT *
    FROM joined
    -- Postgres lets you reference SELECT-list aliases in ORDER BY as
    -- bare identifiers, but NOT inside expressions (``has_paper AND
    -- has_fulltext`` would fail with ``column "has_paper" does not
    -- exist``). Wrapping the joined output in a CTE turns the aliases
    -- into real column names that any expression can see — keeps the
    -- "incomplete trios first" ordering without repeating the JOIN
    -- predicates.
    ORDER BY (has_paper AND has_fulltext AND has_archive) ASC,
             last_touched DESC NULLS LAST
    LIMIT 50
    """
)

### Storage stats

How many papers we've archived, how many bytes that is on disk, and which parser produced each body. The fallback chain is `pymupdf4llm` → `pymupdf` → `pypdf`; the further down a parser sits, the more we should suspect the PDF (image-only, restricted env, malformed). `truncated_count` flags papers whose parsed Markdown exceeded the 1 MiB cap.

In [ ]:
q(
    """
    SELECT
      parser_used,
      count(*) AS papers,
      sum(truncated::int) AS truncated_count,
      round(sum(size_bytes) / 1024.0 / 1024.0, 1) AS total_pdf_mb,
      round(avg(size_bytes) / 1024.0 / 1024.0, 2) AS avg_pdf_mb,
      round(sum(length(parsed_text)) / 1024.0 / 1024.0, 1) AS total_parsed_mb,
      min(archived_at) AS first_archived,
      max(archived_at) AS last_archived
    FROM paper_archives
    GROUP BY parser_used
    ORDER BY papers DESC
    """
)

### Drill into one paper's archive

Pulls the archive provenance, the cc_docs metadata, and a head of the parsed Markdown for the **most recently archived** paper. Eyeball that what BM25 indexes actually reads like prose (not gibberish from an image-only PDF or a bad parser fall-through).

Each cell in this drill-in trio (this query, the parsed-text peek, the PDF dump) re-runs the `ORDER BY archived_at DESC LIMIT 1` selector itself — no shared Python state, so the three are always consistent without any variable to set or stale config to manage. To inspect a specific paper instead of the latest, swap the `chosen` CTE for an `=` predicate on a paperId from the listing two cells up.

The `parsed_text_chars` vs `fulltext_body_chars` delta is the truncation gap — anything > 0 means `paper_archives.parsed_text` has content that didn't make it into the BM25-indexed `company_context_documents.body` (capped at 1 MiB UTF-8).

In [ ]:
archive_row = q(
    """
    WITH chosen AS (
      SELECT paper_id FROM paper_archives ORDER BY archived_at DESC LIMIT 1
    )
    SELECT
      a.paper_id,
      a.source_url,
      a.mime_type,
      a.size_bytes,
      a.parser_used,
      a.truncated,
      a.pdf_sha256,
      length(a.parsed_text) AS parsed_text_chars,
      length(f.body) AS fulltext_body_chars,
      (f.metadata->>'pdfSha256') AS fulltext_metadata_sha,
      (a.pdf_sha256 = f.metadata->>'pdfSha256') AS sha_integrity_ok,
      p.title,
      a.archived_at
    FROM paper_archives a
    JOIN chosen c ON c.paper_id = a.paper_id
    LEFT JOIN company_context_documents p
      ON p.document_id = 'semantic_scholar:paper:' || a.paper_id
    LEFT JOIN company_context_documents f
      ON f.document_id = 'semantic_scholar:paper_fulltext:' || a.paper_id
    """
)
if not len(archive_row):
    print("paper_archives is empty — run an archive workflow first.")
else:
    row = archive_row.iloc[0]
    print(f"paper_id:      {row['paper_id']}")
    print(f"Title:         {row['title']}")
    print(f"Source URL:    {row['source_url']}")
    print(f"Parser:        {row['parser_used']} (truncated={row['truncated']})")
    print(f"PDF:           {row['size_bytes']:,} bytes, sha256={row['pdf_sha256'][:16]}...")
    print(f"Parsed text:   {row['parsed_text_chars']:,} chars in paper_archives")
    print(f"Indexed body:  {row['fulltext_body_chars']:,} chars in cc_docs.paper_fulltext")
    print(f"Sha integrity: {row['sha_integrity_ok']}")
    print(f"Archived:      {row['archived_at']}")
archive_row

### Peek at the parsed Markdown

Writes two files for the most recently archived paper and auto-opens the rendered version in your default browser:

1. **`/tmp/paper_archives/{paper_id}.md`** — raw `paper_archives.parsed_text` (uncapped) written verbatim. Open standalone for diffing or piping into other tools.
2. **`/tmp/paper_archives/{paper_id}.html`** — same content, rendered to HTML via `mistune` and opened via `subprocess.run(["open", ...])` (the only reliable way to bypass Cursor's `file://` link interception in notebook outputs).

Cell output is just the paths + char count — keeps the notebook scrollable. Re-running launches another browser tab; close them as you go.

In [ ]:
import html as html_escape
import subprocess
import sys
from pathlib import Path

import mistune

out_dir = Path("/tmp/paper_archives")
out_dir.mkdir(exist_ok=True)

parsed = q(
    """
    SELECT paper_id, parsed_text
      FROM paper_archives
      ORDER BY archived_at DESC
      LIMIT 1
    """
)
if not len(parsed):
    print("paper_archives is empty")
else:
    paper_id = parsed.iloc[0]["paper_id"]
    text = parsed.iloc[0]["parsed_text"]

    md_path = out_dir / f"{paper_id}.md"
    md_path.write_text(text)

    # Render markdown → HTML via mistune (already in the venv as a
    # JupyterLab transitive dep, so no new requirement). Wrap in a
    # minimal HTML shell with prose-friendly styling so the file is
    # standalone-viewable in any browser.
    body_html = mistune.html(text)
    html_doc = (
        "<!DOCTYPE html><html><head>"
        '<meta charset="utf-8">'
        f"<title>{html_escape.escape(paper_id)}</title>"
        "<style>"
        "body{max-width:840px;margin:2em auto;font-family:-apple-system,"
        "BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;line-height:1.6;"
        "padding:0 1em;color:#24292e}"
        "pre,code{background:#f6f8fa;padding:.2em .4em;border-radius:3px;"
        "font-family:'SF Mono',Menlo,Consolas,monospace;font-size:.9em}"
        "pre{padding:1em;overflow:auto}pre code{background:none;padding:0}"
        "h1,h2,h3{border-bottom:1px solid #eaecef;padding-bottom:.3em}"
        "a{color:#0366d6}img{max-width:100%}"
        "</style></head><body>"
        f"{body_html}"
        "</body></html>"
    )
    html_path = out_dir / f"{paper_id}.html"
    html_path.write_text(html_doc)

    # macOS ``open`` honours Launch Services — .html → default browser
    # (Safari/Chrome/etc.), bypassing Cursor's notebook-output ``file://``
    # link interception. ``check=False`` because the cell still wants to
    # succeed even on Linux/CI where ``open`` isn't present.
    if sys.platform == "darwin":
        subprocess.run(["open", str(html_path)], check=False)

    print(f"paper_id  {paper_id}")
    print(f"md  →     {md_path}")
    print(f"html →    {html_path}  (opening in default browser)")
    print(f"chars     {len(text):,}")

### Export a PDF to disk and open in a browser

Writes `paper_archives.pdf_bytes` (the raw BYTEA column from Postgres — see the storage model above) for the most recently archived paper to `/tmp/paper_archives/{paper_id}.pdf`, verifies SHA-256 + the `%PDF-` magic header, and auto-opens it in Arc (or Chrome, or Safari as a universal fallback). Browsers render PDFs natively — the in-cell-output `<iframe>` approach doesn't work because Cursor's notebook webview locks down iframe sources via CSP, and Cursor's editor has no native PDF viewer.

The cell forces a browser via `open -a "<Browser>"` to override macOS Launch Services (which would otherwise route `.pdf` to Preview). Walks `_browser_candidates` until one succeeds; prepend your preferred browser if you use Firefox/Brave/etc.

In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

out_dir = Path("/tmp/paper_archives")
out_dir.mkdir(exist_ok=True)

pdf = q(
    """
    SELECT paper_id, pdf_bytes, size_bytes, pdf_sha256
      FROM paper_archives
      ORDER BY archived_at DESC
      LIMIT 1
    """
)
if not len(pdf):
    print("paper_archives is empty")
else:
    row = pdf.iloc[0]
    paper_id = row["paper_id"]
    blob = bytes(row["pdf_bytes"])
    out_path = out_dir / f"{paper_id}.pdf"
    out_path.write_bytes(blob)
    on_disk_sha = hashlib.sha256(blob).hexdigest()
    sha_ok = on_disk_sha == row["pdf_sha256"]
    magic_ok = blob[:5] == b"%PDF-"

    # Force the PDF into a browser tab (renders natively). Without
    # ``-a <browser>`` macOS Launch Services routes ``.pdf`` to Preview,
    # and ``-a Cursor`` opens it as a binary buffer (no native viewer).
    # ``open -a`` returns rc=1 when the app isn't installed, so the
    # loop walks the list until one wins. Safari is preinstalled on
    # every macOS, so the loop is guaranteed to terminate there.
    _browser_candidates = ["Arc", "Google Chrome", "Safari"]
    opened_in = None
    if sys.platform == "darwin":
        for browser in _browser_candidates:
            rc = subprocess.run(
                ["open", "-a", browser, str(out_path)],
                check=False,
                capture_output=True,
            ).returncode
            if rc == 0:
                opened_in = browser
                break

    print(f"paper_id  {paper_id}")
    print(f"Wrote     {out_path}")
    if opened_in:
        print(f"Opening   in {opened_in}")
    elif sys.platform == "darwin":
        print(f"Opening   FAILED ({_browser_candidates}); file is still on disk")
    print(f"Size      {len(blob):,} bytes (expected {row['size_bytes']:,})")
    print(f"SHA-256   {on_disk_sha}")
    print(f"sha_ok    {sha_ok}")
    print(f"magic_ok  {magic_ok}  (first 8 bytes: {blob[:8]!r})")

### BM25 over paper bodies (not just abstracts)

Same `body @@@` / `paradedb.score` ranking as the cell at the top, but scoped to `source_type IN ('paper', 'paper_fulltext')` so it searches inside paper full-text in addition to abstracts. Returns the top hits, and a tag column says whether each hit came from the abstract (`paper`) or the parsed full-text (`paper_fulltext`). A query that only matches `paper_fulltext` rows is one the archive flow unlocked that pure-abstract BM25 would have missed.

In [ ]:
q(
    """
    SELECT
      source_type,
      left(title, 60) AS title,
      paradedb.score(document_id) AS score,
      (metadata->>'year')::int AS year,
      left(body, 240) AS preview
    FROM company_context_documents
    WHERE source = 'semantic_scholar'
      AND source_type IN ('paper', 'paper_fulltext')
      AND body @@@ %s
    ORDER BY score DESC
    LIMIT 10
    """,
    ("attention mechanism transformer",),
)